# Functions for cross tabulated format (CTF) 1

Created: 2026-11-8

The previous notebook in this series [Non-pivot xslx functions 3](non-pivot-xlsx-03.ipynb) worked through functionality for cross tabulated format sheets in the [December 2014 statistics for foreign residents in Japan](http://www.e-stat.go.jp/SG1/estat/List.do?lid=000001133760) using tree data structures for parsing the stub column and column headers. This notebook is for adding this new functionality to the `excel` module. Here I use tables several years in the collection in order to cover variability in their formats. 

## Install the necessary packages

In [ ]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install numpy
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl
%pip install treelib

## Install the newest version of estatjp

Repeat during development to import revisions.

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [2]:
url20120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104290&fileKind=4"
url20120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104291&fileKind=0" # column-oriented
url20120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104292&fileKind=4"
url20120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104293&fileKind=0" # column-oriented
url201203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104294&fileKind=4"
url201204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104295&fileKind=4"
url201205 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104296&fileKind=4"

## Import the module and download a file

In [ ]:
from estatjp.xlsx import xlsx
from pathlib import Path

theurl = url201203
dataset = "201212"
ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa") / dataset
if Path.exists(dataraw) == False:
    Path.mkdir(dataraw)
dataprocessed = ROOT_DIR / Path(".data/processed/isa") / dataset
if Path.exists(dataprocessed) == False:
    Path.mkdir(dataprocessed)
newdata = xlsx.download_to_folder(theurl, dataraw, overwrite=True, verbose=True)
print('source: ' + str(newdata))
if newdata != None:
    processeddata = dataprocessed / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Date': 'Sun, 08 Nov 2026 08:00:17 GMT', 'Strict-Transport-Security': 'max-age=0', 'X-Zen-Fury': '5d9c53c37e5138a7204e76430c853ea01ee39767', 'Server': 'ZENEDGE', 'X-XSS-Protection': '1; mode=block', 'Content-Disposition': "attachment; filename*=UTF-8''20-12-03.xlsx", 'X-Frame-Options': 'SAMEORIGIN', 'X-Cache-Status': 'NOTCACHED', 'X-Cache': 'MISS', 'Accept-Ranges': 'bytes', 'Cache-Control': 'private', 'X-Content-Type-Options': 'nosniff', 'Content-Encoding': 'gzip', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\201212\20-12-03.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\201212\20-12-03.csv


## Examine the worksheets and look for pivot tables or column-orientation

There are four known dataformats among the xlsx files
from E-Stat: cross tabulated format (CTF), column-oriented format
(COF), pivot (PVT), and pivot but no cache records. 
Determine format from file.

In [ ]:
from estatjp.xlsx.xlsx import get_dataformat

dataformat = get_dataformat(newdata, verbose=True)
print('data format: ' + dataformat)

['20-12-03']
title: 20-12-03
dimensions: A1:FJ45
print_title_cols: $A:$A
print_title_rows: $4:$5
_pivots len: 0
merged_cells len: 0
tableno: 03
data format: CTF
